# PyTorch 自定义数据处理与变换

> 本笔记本是 [自定义数据处理.ipynb](./自定义数据处理.ipynb) 的 **PyTorch 等价版本**，
> 原版使用 Keras 预处理层实现数据转换，本版使用 PyTorch 自定义变换实现相同功能。

本教程深入讲解 PyTorch 中自定义数据处理的实现方法，包括变换函数设计、torchvision.transforms 图像处理、以及即时预处理与预处理的性能权衡。

## 学习目标
1. 掌握自定义 Dataset 类中集成 transforms 的方法
2. 学会使用 torchvision.transforms 处理图像数据
3. 理解 Compose 组合变换与自定义变换函数
4. 对比即时预处理与预处理数据的性能权衡

## 1. 环境配置

In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch版本: {torch.__version__}")
print(f"torchvision版本: {torchvision.__version__}")
print(f"计算设备: {device}")

## 2. 自定义 Dataset 与 Transforms

### 2.1 Transform 的设计模式

PyTorch 中变换(transform)的核心设计模式是将变换函数作为参数传入 Dataset，
在 `__getitem__` 中对数据应用变换。这种模式将数据加载与预处理解耦。

**核心接口**:
- `transform`: 特征变换函数，在 `__getitem__` 中对输入数据调用
- `target_transform`: 标签变换函数，对标签调用
- 变换函数可以是任意可调用对象（函数、类实例、lambda）

In [ ]:
class CustomDataset(Dataset):
    """
    支持变换函数的自定义数据集
    Custom Dataset class with transform support.

    Parameters:
    -----------
    features : numpy.ndarray
        特征矩阵 / Feature matrix
    labels : numpy.ndarray
        标签数组 / Label array
    transform : callable, optional
        特征变换函数 / Feature transform function
    target_transform : callable, optional
        标签变换函数 / Label transform function
    """

    def __init__(self, features, labels, transform=None, target_transform=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        """返回数据集大小 / Return dataset size"""
        return len(self.features)

    def __getitem__(self, idx):
        """
        获取样本并应用变换
        Get sample by index and apply transforms.
        """
        x = self.features[idx]
        y = self.labels[idx]

        if self.transform:
            x = self.transform(x)
        if self.target_transform:
            y = self.target_transform(y)

        return x, y


# 创建测试数据
np.random.seed(RANDOM_SEED)
test_features = np.random.randn(100, 5).astype(np.float32)
test_labels = np.random.randint(0, 3, size=100)

# 不使用变换
ds_no_transform = CustomDataset(test_features, test_labels)
x, y = ds_no_transform[0]
print(f"无变换: 特征={x[:3]}, 标签={y}")

# 使用 lambda 变换
ds_lambda = CustomDataset(
    test_features, test_labels,
    transform=lambda x: (x - x.mean()) / (x.std() + 1e-8)
)
x_t, y_t = ds_lambda[0]
print(f"lambda变换: 特征={x_t[:3]}, 标签={y_t}")

### 2.2 自定义数值变换

创建可复用的变换类，实现 `__call__` 方法。
这种方式比 lambda 更灵活，可以保存参数和状态。

In [ ]:
class CustomStandardization:
    """
    自定义 Z-score 标准化变换
    Custom Z-score standardization transform.

    Formula: z = (x - mean) / std

    Attributes:
    -----------
    means : numpy.ndarray
        特征均值 / Feature means
    stds : numpy.ndarray
        特征标准差 / Feature standard deviations
    """

    def __init__(self, epsilon=1e-7):
        self.epsilon = epsilon
        self.means = None
        self.stds = None

    def fit(self, data):
        """
        从数据中学习均值和标准差
        Learn means and stds from data.

        Parameters:
        -----------
        data : numpy.ndarray
            训练数据 / Training data
        """
        data = np.asarray(data, dtype=np.float32)
        self.means = np.mean(data, axis=0, keepdims=True)
        self.stds = np.std(data, axis=0, keepdims=True)
        self.stds = np.where(self.stds < self.epsilon, 1.0, self.stds)

    def __call__(self, x):
        """应用标准化 / Apply standardization"""
        if self.means is None:
            raise RuntimeError("变换未初始化，请先调用 fit()")
        x_np = x.numpy() if isinstance(x, torch.Tensor) else x
        return torch.tensor((x_np - self.means.flatten()) / self.stds.flatten(), dtype=torch.float32)

    def inverse_transform(self, normalized):
        """逆变换 / Inverse transform"""
        return normalized * torch.tensor(self.stds.flatten()) + torch.tensor(self.means.flatten())


class CustomMinMaxNormalization:
    """
    自定义 Min-Max 归一化变换
    Custom Min-Max normalization transform.

    Formula: x_norm = (x - min) / (max - min)
    """

    def __init__(self, feature_range=(0, 1), epsilon=1e-7):
        self.feature_range = feature_range
        self.epsilon = epsilon
        self.data_min = None
        self.data_max = None

    def fit(self, data):
        """从数据中学习最小值和最大值 / Learn min and max from data."""
        data = np.asarray(data, dtype=np.float32)
        self.data_min = np.min(data, axis=0, keepdims=True)
        self.data_max = np.max(data, axis=0, keepdims=True)

    def __call__(self, x):
        """应用归一化 / Apply normalization"""
        if self.data_min is None:
            raise RuntimeError("变换未初始化，请先调用 fit()")
        x_np = x.numpy() if isinstance(x, torch.Tensor) else x
        data_range = self.data_max.flatten() - self.data_min.flatten()
        data_range = np.where(data_range < self.epsilon, 1.0, data_range)
        scaled = (x_np - self.data_min.flatten()) / data_range
        min_val, max_val = self.feature_range
        return torch.tensor(scaled * (max_val - min_val) + min_val, dtype=torch.float32)


# 测试自定义标准化
standardizer = CustomStandardization()
standardizer.fit(test_features)

ds_std = CustomDataset(test_features, test_labels, transform=standardizer)
x_std, _ = ds_std[0]
print(f"标准化后样本0: {x_std.numpy()}")
print(f"均值: {x_std.mean():.6f}, 标准差: {x_std.std():.4f}")

# 测试自定义归一化
normalizer = CustomMinMaxNormalization(feature_range=(-1, 1))
normalizer.fit(test_features)

ds_norm = CustomDataset(test_features, test_labels, transform=normalizer)
x_norm, _ = ds_norm[0]
print(f"\n归一化到[-1,1]后样本0: {x_norm.numpy()}")
print(f"范围: [{x_norm.min():.4f}, {x_norm.max():.4f}]")

### 2.3 变换函数作为可调用对象

PyTorch 的变换遵循可调用对象(callable)协议。任何实现了 `__call__` 方法的对象都可以作为变换。

**三种变换形式**:
1. **函数**: 最简单，适用于无状态变换
2. **Lambda**: 匿名函数，适用于简单一次性变换
3. **类实例**: 最灵活，可以保存状态和参数

In [ ]:
# 1. 函数形式 / Function form
def normalize_per_sample(x):
    """逐样本标准化 / Per-sample standardization"""
    return (x - x.mean()) / (x.std() + 1e-8)

# 2. Lambda 形式 / Lambda form
normalize_scale = lambda x: x * 0.01  # 简单缩放 / Simple scaling

# 3. 类形式 / Class form (已在上面的 CustomStandardization 演示)

# 对比三种形式
ds_func = CustomDataset(test_features, test_labels, transform=normalize_per_sample)
ds_lambda2 = CustomDataset(test_features, test_labels, transform=normalize_scale)
ds_class = CustomDataset(test_features, test_labels, transform=standardizer)

x_func, _ = ds_func[0]
x_lamb, _ = ds_lambda2[0]
x_cls, _ = ds_class[0]

print("三种变换形式对比:")
print(f"  函数形式: {x_func[:3].numpy()}")
print(f"  Lambda形式: {x_lamb[:3].numpy()}")
print(f"  类形式: {x_cls[:3].numpy()}")

# Compose 多个变换: 先标准化再缩放
class ComposeTransforms:
    """
    组合多个变换
    Compose multiple transforms into one.
    """

    def __init__(self, transforms_list):
        self.transforms_list = transforms_list

    def __call__(self, x):
        for t in self.transforms_list:
            x = t(x)
        return x


# 使用自定义 Compose
custom_compose = ComposeTransforms([standardizer, normalize_scale])
ds_compose = CustomDataset(test_features, test_labels, transform=custom_compose)
x_comp, _ = ds_compose[0]
print(f"\nCompose(标准化, 缩放): {x_comp[:3].numpy()}")

## 3. torchvision.transforms 图像变换

### 3.1 常用图像变换

`torchvision.transforms` 提供了丰富的图像变换操作：
- `ToTensor`: PIL Image/numpy → tensor，像素值 [0,255] → [0,1]
- `Resize`: 调整图像尺寸
- `CenterCrop` / `RandomCrop`: 裁剪
- `Normalize(mean, std)`: 逐通道标准化

In [ ]:
from PIL import Image

# 创建合成图像用于演示 / Create synthetic images for demo
np.random.seed(RANDOM_SEED)
sample_images = [Image.fromarray(
    np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
) for _ in range(4)]

print(f"原始图像: 尺寸={sample_images[0].size}, 模式={sample_images[0].mode}")

# ToTensor: PIL → Tensor, [0,255] → [0,1]
to_tensor = transforms.ToTensor()
img_tensor = to_tensor(sample_images[0])
print(f"\nToTensor 后: 形状={img_tensor.shape}, dtype={img_tensor.dtype}")
print(f"值域: [{img_tensor.min():.3f}, {img_tensor.max():.3f}]")

# Resize
resize = transforms.Resize((32, 32))
resized = resize(sample_images[0])
print(f"\nResize 后: 尺寸={resized.size}")

# CenterCrop
center_crop = transforms.CenterCrop(48)
cropped = center_crop(sample_images[0])
print(f"CenterCrop 后: 尺寸={cropped.size}")

# Normalize (需要先转为 Tensor)
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],  # ImageNet 均值
    std=[0.229, 0.224, 0.225]    # ImageNet 标准差
)
normalized_img = normalize(img_tensor)
print(f"\nNormalize 后: 均值={normalized_img.mean(dim=[1,2]).tolist()}")
print(f"  值域: [{normalized_img.min():.3f}, {normalized_img.max():.3f}]")

### 3.2 数据增强变换

数据增强通过对训练数据施加随机变换来增加数据多样性，
是防止过拟合的重要手段。

**常用增强变换**:
- `RandomHorizontalFlip`: 随机水平翻转
- `RandomRotation`: 随机旋转
- `ColorJitter`: 颜色抖动（亮度、对比度、饱和度、色调）
- `RandomResizedCrop`: 随机裁剪并缩放

In [ ]:
# 数据增强演示 / Data augmentation demo
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop(size=32, scale=(0.8, 1.0)),
    transforms.ToTensor(),
])

# 对同一图像应用多次增强 / Apply augmentation multiple times to same image
print("同一图像的5次增强结果:")
for i in range(5):
    augmented = aug_transform(sample_images[0])
    print(f"  第{i+1}次: 形状={augmented.shape}, 均值={augmented.mean():.4f}")

### 3.3 transforms.Compose 组合变换

`transforms.Compose` 将多个变换串联成流水线，数据依次通过每个变换。

**训练与验证的不同流水线**:
- **训练**: 数据增强 + 标准化（增加泛化能力）
- **验证/测试**: 仅标准化（保证评估一致性）

In [ ]:
# ImageNet 标准化参数 / ImageNet normalization parameters
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# 训练集变换流水线：增强 + 标准化
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# 验证集变换流水线：仅标准化，不做增强
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

print("训练集变换:")
for i, t in enumerate(train_transform.transforms):
    print(f"  {i+1}. {t.__class__.__name__}")

print("\n验证集变换:")
for i, t in enumerate(val_transform.transforms):
    print(f"  {i+1}. {t.__class__.__name__}")

## 4. 完整图像数据集示例

### 4.1 自定义图像数据集

创建一个从内存或磁盘加载图像的自定义 Dataset 类，
支持不同的训练/验证变换流水线。

In [ ]:
class SyntheticImageDataset(Dataset):
    """
    合成图像数据集（演示用）
    Synthetic image dataset for demonstration.

    Parameters:
    -----------
    num_samples : int
        样本数量 / Number of samples
    num_classes : int
        类别数量 / Number of classes
    transform : callable, optional
        图像变换 / Image transform
    """

    def __init__(self, num_samples=200, num_classes=10, transform=None):
        self.num_samples = num_samples
        self.num_classes = num_classes
        self.transform = transform

        # 生成合成图像和标签 / Generate synthetic images and labels
        np.random.seed(RANDOM_SEED)
        self.images = [
            Image.fromarray(np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8))
            for _ in range(num_samples)
        ]
        self.labels = np.random.randint(0, num_classes, num_samples).tolist()

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        """
        获取图像样本并应用变换
        Get image sample and apply transform.
        """
        image = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


# 创建训练集和验证集（使用不同变换）
train_img_ds = SyntheticImageDataset(num_samples=200, transform=train_transform)
val_img_ds = SyntheticImageDataset(num_samples=50, transform=val_transform)

train_img_loader = DataLoader(train_img_ds, batch_size=16, shuffle=True)
val_img_loader = DataLoader(val_img_ds, batch_size=16, shuffle=False)

# 验证一个批次
for images, labels in train_img_loader:
    print(f"训练批次: 图像={images.shape}, 标签={labels.shape}")
    print(f"图像值域: [{images.min():.3f}, {images.max():.3f}]")
    break

### 4.2 使用 torchvision.datasets

`torchvision.datasets` 提供了常用数据集的便捷加载方式，
内置对 transforms 的支持。

In [ ]:
# 使用 CIFAR-10 数据集（需要下载）
# Use CIFAR-10 dataset (requires download)

# 定义变换 / Define transforms
cifar_train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]),
])

cifar_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                         std=[0.2470, 0.2435, 0.2616]),
])

# 注意: 实际使用时取消注释
# train_dataset = torchvision.datasets.CIFAR10(
#     root='./data', train=True, download=True, transform=cifar_train_transform
# )
# test_dataset = torchvision.datasets.CIFAR10(
#     root='./data', train=False, download=True, transform=cifar_test_transform
# )
# train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("torchvision.datasets 常用数据集:")
print("  - CIFAR10 / CIFAR100")
print("  - MNIST / FashionMNIST")
print("  - ImageFolder (自定义图像目录)")
print("  - DatasetFolder (自定义文件目录)")
print("\n所有数据集都支持 transform 参数")

### 4.3 训练与验证的不同变换流水线

演示完整的图像分类训练流程，训练集使用数据增强，验证集仅标准化。

In [ ]:
class SimpleCNN(nn.Module):
    """
    简单的 CNN 分类模型
    A simple CNN classification model.

    Parameters:
    -----------
    num_classes : int
        分类数 / Number of classes
    """

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 56 * 56, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


model = SimpleCNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"模型参数数: {sum(p.numel() for p in model.parameters())}")

# 训练几个 epoch
EPOCHS = 3
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in train_img_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)

    train_acc = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} - loss: {total_loss/total:.4f} - acc: {train_acc:.4f}")

## 5. 即时预处理 vs 预处理数据

### 5.1 即时预处理 (On-the-fly)

在 `__getitem__` 中每次访问时都应用变换。

**优点**:
- 灵活性高，可以随时更换变换
- 支持数据增强（每次生成不同的变换结果）
- 存储开销小

**缺点**:
- 每个 epoch 重复计算相同的变换（非增强部分）
- CPU 成为潜在瓶颈

In [ ]:
class OnTheFlyDataset(Dataset):
    """
    即时预处理数据集
    Dataset with on-the-fly preprocessing.
    """

    def __init__(self, features, labels, transform=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        x = self.features[idx]
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)  # 每次访问都重新计算
        return x, y


# 创建即时预处理数据集
np.random.seed(RANDOM_SEED)
large_features = np.random.randn(5000, 50).astype(np.float32)
large_labels = np.random.randint(0, 5, 5000)

# 标准化变换（每次访问都计算）
standardizer = CustomStandardization()
standardizer.fit(large_features)

otf_ds = OnTheFlyDataset(large_features, large_labels, transform=standardizer)
otf_loader = DataLoader(otf_ds, batch_size=64, shuffle=True)

# 计时
start = time.time()
for epoch in range(5):
    for x, y in otf_loader:
        pass
otf_time = time.time() - start
print(f"即时预处理 (5 epochs): {otf_time:.4f}s")

### 5.2 预处理数据 (Preprocessed)

提前对所有数据应用变换，将结果保存，训练时直接加载。

**优点**:
- 训练速度更快（无需重复计算）
- 不受 CPU 瓶颈限制

**缺点**:
- 不支持数据增强（增强需要随机性）
- 需要更多存储空间
- 变换逻辑变更时需要重新预处理

In [ ]:
# 预处理所有数据 / Preprocess all data in advance
features_tensor = torch.tensor(large_features, dtype=torch.float32)
labels_tensor = torch.tensor(large_labels, dtype=torch.long)

# 一次性应用标准化 / Apply standardization once
means = torch.tensor(standardizer.means.flatten(), dtype=torch.float32)
stds = torch.tensor(standardizer.stds.flatten(), dtype=torch.float32)
preprocessed_features = (features_tensor - means) / stds

# 创建预处理后的数据集
pre_ds = TensorDataset(preprocessed_features, labels_tensor)
pre_loader = DataLoader(pre_ds, batch_size=64, shuffle=True)

# 计时
start = time.time()
for epoch in range(5):
    for x, y in pre_loader:
        pass
pre_time = time.time() - start
print(f"预处理数据 (5 epochs): {pre_time:.4f}s")
print(f"\n速度提升: {otf_time / pre_time:.2f}x")

### 5.3 性能对比

| 策略 | 适用场景 | 优势 | 劣势 |
|------|----------|------|------|
| 即时预处理 | 数据增强、快速实验 | 灵活、节省存储 | CPU 开销 |
| 预处理数据 | 固定变换、大规模训练 | 速度快 | 不支持增强、需额外存储 |

**建议**:
- **训练集**: 使用即时预处理 + 数据增强
- **验证/测试集**: 可以预处理（无随机性）
- **大规模训练**: 将增强也预计算为多个版本

In [ ]:
print("=" * 50)
print("性能对比总结")
print("=" * 50)
print(f"即时预处理 (5 epochs): {otf_time:.4f}s")
print(f"预处理数据   (5 epochs): {pre_time:.4f}s")
print(f"加速比: {otf_time / pre_time:.2f}x")
print()
print("注意: 对于简单变换(如标准化)，加速效果有限。")
print("对于复杂变换(如图像增强、特征提取)，加速效果更显著。")

## 小结

### PyTorch 变换核心概念

| 概念 | 说明 |
|------|------|
| `transform` | Dataset 中应用的特征变换 |
| `target_transform` | 标签变换 |
| `transforms.Compose` | 组合多个变换 |
| `transforms.ToTensor` | PIL/numpy → tensor |
| `transforms.Normalize` | 逐通道标准化 |
| 自定义变换类 | 实现 `__call__` 方法 |

### 最佳实践

1. **训练集**: 使用数据增强 + 标准化
2. **验证/测试集**: 仅标准化，保证一致性
3. **标准化参数**: 仅从训练集计算
4. **变换位置**: PyTorch 在 Dataset 中变换，TF 在模型层中变换
5. **性能**: 简单变换即时处理，复杂特征提取可预处理

## TF vs PyTorch 对照

| 概念 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 标准化层 | `layers.Normalization()` + `adapt()` | 自定义 `CustomStandardization` + `fit()` |
| 缩放层 | `layers.Rescaling(scale=1/255)` | `transforms.ToTensor()` (自动 0-1) + `transforms.Normalize()` |
| 预处理位置 | 模型内部（Keras 层） | Dataset 的 `__getitem__` 中 |
| 预处理与模型关系 | 可集成到模型中，训练推理一致 | 与模型分离，需手动确保一致性 |
| 字符串编码 | `layers.StringLookup` + `CategoryEncoding` | 自定义 Vocabulary 映射 |
| 类别嵌入 | `layers.Embedding` | `nn.Embedding` |
| 图像增强 | `tf.image` 函数 | `torchvision.transforms` |
| 变换组合 | `tf.keras.Sequential([layer1, layer2])` | `transforms.Compose([t1, t2])` |
| 训练/推理一致性 | 预处理层在模型内，天然一致 | 需要分别定义 train/val transform |
| 模型保存 | `model.save()` 包含预处理参数 | 仅保存模型权重，变换需单独管理 |
| GPU 加速 | 预处理层可在 GPU 执行 | 变换通常在 CPU 执行（DataLoader 的 worker） |

## 练习

### 练习1：创建高斯噪声变换

实现一个添加高斯噪声的自定义变换类：
```python
class GaussianNoise:
    def __init__(self, mean=0.0, std=0.1):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        noise = torch.randn_like(x) * self.std + self.mean
        return x + noise
```
思考：高斯噪声变换应该用于训练集还是验证集？为什么？

### 练习2：构建完整的图像分类流水线

使用 CIFAR-10 数据集，构建包含以下组件的完整分类流水线：
```python
# 训练集增强流水线
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
```
思考：比较使用和不使用数据增强时模型的验证准确率差异。

### 练习3：对比即时预处理与预处理的训练速度

创建一个较复杂的数据集和变换（如多步特征工程），
测量即时预处理 vs 预处理数据在 10 个 epoch 中的训练时间差异：
```python
class ComplexTransform:
    def __init__(self, means, stds):
        self.means = means
        self.stds = stds

    def __call__(self, x):
        # 第1步: 标准化
        x = (x - self.means) / self.stds
        # 第2步: 特征交叉
        x_cross = x[:len(x)//2] * x[len(x)//2:]
        # 第3步: 拼接
        return torch.cat([x, x_cross])
```
思考：变换复杂度如何影响即时预处理与预处理的性能差距？